# 확률론 3주차 과제 4 — 타이타닉 승객 나이의 PDF와 CDF

---

**과목**: 확률론  
**주차**: 3주차  
**주제**: 연속 확률변수의 확률밀도함수(PDF)와 누적분포함수(CDF) — 타이타닉 데이터셋  

---

## 학습 목표

1. 연속 확률변수와 이산 확률변수의 차이를 이해한다
2. 확률밀도함수(PDF)의 의미와 해석을 정확히 학습한다
3. 히스토그램과 KDE를 통한 PDF 추정 방법을 익힌다
4. CDF를 구성하고 실제 확률 계산에 활용한다
5. KDE 대역폭(bandwidth)이 추정에 미치는 영향을 분석한다

## 기본 용어 정리

### 연속 확률변수 (Continuous Random Variable)

어떤 구간 내의 **모든 실수값**을 취할 수 있는 확률변수이다.

| 구분 | 이산 확률변수 | 연속 확률변수 |
|------|-------------|-------------|
| 값의 종류 | 셀 수 있음 (countable) | 셀 수 없음 (uncountable) |
| 확률 함수 | PMF: $P(X=x)$ | PDF: $f(x)$ |
| 특정값 확률 | $P(X=x) > 0$ 가능 | $P(X=x) = 0$ (항상!) |
| 구간 확률 | $\sum_{k=a}^{b} P(X=k)$ | $\int_a^b f(x)dx$ |
| CDF | 계단함수 | 연속함수 |

### 확률밀도함수 (PDF: Probability Density Function)

연속 확률변수 $X$의 확률분포를 나타내는 함수 $f(x)$:

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx$$

**핵심 주의사항**: $f(x)$는 확률이 **아니다**! $f(x)$는 **밀도(density)**이다.

**PDF의 성질:**
1. $f(x) \geq 0$ (모든 $x$에 대해)
2. $\int_{-\infty}^{+\infty} f(x) \, dx = 1$ (전체 면적 = 1)
3. $f(x)$의 값은 1보다 클 수 있다! (밀도이므로)

### 커널 밀도 추정 (KDE: Kernel Density Estimation)

히스토그램의 연속적 대안. 각 데이터 포인트에 커널(보통 가우시안)을 놓고 합산하여 부드러운 밀도 함수를 추정한다.

$$\hat{f}(x) = \frac{1}{n \cdot h} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

여기서:
- $K$: 커널 함수 (보통 가우시안)
- $h$: **대역폭(bandwidth)** — KDE의 핵심 파라미터
- $n$: 데이터 수

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon
from itertools import product
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

---

## 1단계: 데이터 로딩 및 탐색

타이타닉 데이터셋은 1912년 타이타닉호 승객 891명의 정보를 담고 있다.  
이 중 **age(나이)** 변수를 연속 확률변수로 다룬다.

In [ ]:
# 데이터 로딩
titanic = sns.load_dataset('titanic')

print("=" * 60)
print("타이타닉 데이터셋 기본 정보")
print("=" * 60)
print(f"전체 행 수: {len(titanic)}")
print(f"\n컬럼 목록:")
print(titanic.dtypes)
print(f"\n처음 5행:")
titanic.head()

In [ ]:
# age 변수 탐색
print("=" * 60)
print("age 변수 기초 통계량")
print("=" * 60)

# 결측치 확인
n_total = len(titanic)
n_missing = titanic['age'].isna().sum()
n_valid = n_total - n_missing

print(f"\n결측치 현황:")
print(f"  전체 데이터: {n_total}개")
print(f"  결측치: {n_missing}개 ({n_missing/n_total*100:.1f}%)")
print(f"  유효 데이터: {n_valid}개 ({n_valid/n_total*100:.1f}%)")

# 결측치 제거
age = titanic['age'].dropna()

print(f"\n기초 통계량 (결측치 제거 후):")
print(f"  평균 (mean): {age.mean():.2f}세")
print(f"  중앙값 (median): {age.median():.2f}세")
print(f"  표준편차 (std): {age.std():.2f}세")
print(f"  최솟값 (min): {age.min():.2f}세")
print(f"  최댓값 (max): {age.max():.2f}세")
print(f"  사분위수:")
print(f"    Q1 (25%): {age.quantile(0.25):.2f}세")
print(f"    Q2 (50%): {age.quantile(0.50):.2f}세")
print(f"    Q3 (75%): {age.quantile(0.75):.2f}세")
print(f"  왜도 (skewness): {age.skew():.4f}")
print(f"  첨도 (kurtosis): {age.kurtosis():.4f}")

### 해석

- **결측치**: age 변수에 약 20%의 결측치가 존재한다. 분석에서는 결측치를 제거(listwise deletion)하고 진행한다.
- **분포 특성**: 평균(약 30세)이 중앙값과 비슷하지만, 양의 왜도가 있어 오른쪽으로 약간 치우친 분포이다.
- age는 연속적인 값을 가지므로 **연속 확률변수**로 다루는 것이 적절하다.

---

## 2단계: 히스토그램 + KDE (PDF 추정)

### 히스토그램의 역할

히스토그램은 데이터의 분포를 시각적으로 파악하는 가장 기본적인 도구이다.  
`stat='density'`로 설정하면 y축이 **밀도**가 되어 히스토그램의 총 면적이 1이 된다.

### KDE의 역할

KDE는 히스토그램의 "부드러운 버전"으로, 연속적인 밀도 함수를 추정한다.  
히스토그램은 bin의 선택에 민감하지만, KDE는 대역폭(bandwidth)으로 부드러움을 조절한다.

In [ ]:
# Histogram + KDE 시각화
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(age, bins=30, stat='density', kde=True, color='#3498db',
             edgecolor='white', linewidth=1, alpha=0.6, ax=ax,
             line_kws={'linewidth': 3, 'label': 'KDE (밀도 추정)'})

# 평균과 중앙값 표시
ax.axvline(age.mean(), color='red', linestyle='--', linewidth=2, label=f'평균 = {age.mean():.1f}세')
ax.axvline(age.median(), color='green', linestyle='-.', linewidth=2, label=f'중앙값 = {age.median():.1f}세')

ax.set_xlabel('나이 (Age)', fontsize=14)
ax.set_ylabel('밀도 (Density)', fontsize=14)
ax.set_title('타이타닉 승객 나이 분포: 히스토그램 + KDE\n'
             'y축은 확률이 아닌 "밀도"임에 주의!', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)

# y축 설명 추가
ax.annotate('밀도(density):\n확률이 아닌\n단위 구간당 확률', xy=(60, 0.035),
            fontsize=11, color='gray',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

### 해석

- **히스토그램**: 막대의 높이는 **밀도(density)**이며, 막대의 **넓이 x 높이 = 해당 구간의 확률**이다.
- **KDE 곡선**: 히스토그램을 부드럽게 연결한 연속 곡선. 이것이 PDF의 추정치이다.
- 분포의 특징:
  - 20~30대에 가장 많은 승객이 분포
  - 오른쪽 꼬리가 길다 (양의 왜도)
  - 영유아(0~5세) 구간에도 작은 봉우리 존재

---

## 3단계: Bins 변경 비교 — "같은 데이터, 다른 bins"

히스토그램에서 **bins의 수**는 분포의 형태를 크게 바꿀 수 있다.  
이것은 히스토그램의 주요 한계점이며, KDE가 선호되는 이유이기도 하다.

In [ ]:
# Bins 변경 비교: 10, 20, 50
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bin_counts = [10, 20, 50]

for ax, n_bins in zip(axes, bin_counts):
    sns.histplot(age, bins=n_bins, stat='density', kde=True, color='#3498db',
                 edgecolor='white', alpha=0.6, ax=ax,
                 line_kws={'linewidth': 2.5, 'color': '#e74c3c'})
    
    ax.set_xlabel('나이', fontsize=12)
    ax.set_ylabel('밀도', fontsize=12)
    ax.set_title(f'bins = {n_bins}', fontsize=15, fontweight='bold')
    ax.set_ylim(0, 0.05)

fig.suptitle('같은 데이터, 다른 bins → 히스토그램 형태가 달라진다!\n'
             'KDE(빨간 곡선)는 bins에 무관하게 안정적',
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

### 해석

- **bins=10 (너무 적음)**: 분포의 세부 구조가 묻혀버린다. 과소적합(underfitting).
- **bins=20 (적당)**: 대략적인 분포 형태를 잘 보여준다.
- **bins=50 (너무 많음)**: 잡음(noise)이 강조되어 분포의 일반적 패턴을 파악하기 어렵다.
- **KDE(빨간 곡선)**: bins 선택에 무관하게 거의 동일한 형태를 유지한다 → 더 안정적인 PDF 추정치.

---

## 4단계: PDF 해석 — y축은 확률이 아닌 밀도!

### 가장 흔한 오해

> "PDF의 y축 값이 0.04이면, 그 나이가 될 확률이 4%이다" → **틀림!**

### 올바른 해석

- **밀도(density)** = 단위 구간당 확률의 비율
- **실제 확률** = 밀도 $\times$ 구간 폭

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx \approx f(x) \times (b - a) \quad (\text{좁은 구간에서})$$

### 전체 면적 = 1 확인

$$\int_{-\infty}^{+\infty} f(x) \, dx = 1$$

In [ ]:
# PDF의 올바른 해석: 밀도 × 구간폭 = 확률
from scipy.stats import gaussian_kde

# KDE 추정
kde = gaussian_kde(age)
x_grid = np.linspace(age.min() - 5, age.max() + 5, 1000)
pdf_values = kde(x_grid)

fig, ax = plt.subplots(figsize=(12, 6))

# PDF 곡선
ax.plot(x_grid, pdf_values, color='#3498db', linewidth=2.5, label='PDF (KDE 추정)')

# P(20 <= age <= 30) 영역 표시
mask = (x_grid >= 20) & (x_grid <= 30)
ax.fill_between(x_grid[mask], pdf_values[mask], alpha=0.4, color='#e74c3c',
                label='P(20 ≤ age ≤ 30)')

# 실제 확률 계산
# 방법 1: KDE 적분
prob_20_30 = kde.integrate_box_1d(20, 30)
# 방법 2: 실제 데이터에서 비율
empirical_prob = ((age >= 20) & (age <= 30)).mean()

ax.annotate(f'P(20 ≤ age ≤ 30)\n= 이 영역의 면적\n= {prob_20_30:.4f} ({prob_20_30*100:.1f}%)',
            xy=(25, 0.02), xytext=(45, 0.035),
            fontsize=13, fontweight='bold', color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='#e74c3c'))

ax.set_xlabel('나이 (Age)', fontsize=14)
ax.set_ylabel('밀도 f(x)', fontsize=14)
ax.set_title('PDF에서 확률 = 면적 (밀도의 적분)', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"P(20 ≤ age ≤ 30):")
print(f"  KDE 적분값: {prob_20_30:.4f}")
print(f"  실제 데이터 비율: {empirical_prob:.4f}")

In [ ]:
# 전체 면적 = 1 확인
from scipy.integrate import quad

total_area, error = quad(kde, age.min() - 10, age.max() + 10)

print("=" * 50)
print("PDF 성질 검증: 전체 면적 = 1")
print("=" * 50)
print(f"\n적분 범위: [{age.min()-10:.0f}, {age.max()+10:.0f}]")
print(f"전체 면적 (KDE 적분): {total_area:.6f}")
print(f"1과의 차이: {abs(1 - total_area):.8f}")
print(f"\n결론: 전체 면적 ≈ 1 확인됨 (PDF의 정규성 성질)")

### 해석

- PDF 곡선 아래의 **특정 구간 면적** = 해당 구간의 **확률**이다.
- 예: 20세에서 30세 사이에 해당하는 승객의 비율은 약 36%이다.
- PDF 곡선 아래의 **전체 면적** = 1 (확인 완료).
- y축의 값(밀도)은 1을 초과할 수도 있다! 예를 들어, 0~1 범위만 갖는 변수의 PDF에서 $f(0.5) = 3$이어도 문제없다.

---

## 5단계: 누적분포함수 (CDF) 구성

연속 확률변수의 CDF:

$$F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t) \, dt$$

CDF에서 바로 읽을 수 있는 정보:
- $P(X \leq x) = F(x)$
- $P(X > x) = 1 - F(x)$
- $P(a \leq X \leq b) = F(b) - F(a)$
- **분위수(quantile)**: $F(x) = p$를 만족하는 $x$ 값

In [ ]:
# CDF 시각화: ECDF + KDE 기반 CDF
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: PDF (KDE)
axes[0].plot(x_grid, pdf_values, color='#3498db', linewidth=2.5)
axes[0].fill_between(x_grid, pdf_values, alpha=0.2, color='#3498db')
axes[0].set_xlabel('나이', fontsize=13)
axes[0].set_ylabel('밀도 f(x)', fontsize=13)
axes[0].set_title('PDF (확률밀도함수)', fontsize=16, fontweight='bold')

# 오른쪽: ECDF
age_sorted = np.sort(age)
ecdf_y = np.arange(1, len(age_sorted) + 1) / len(age_sorted)

axes[1].step(age_sorted, ecdf_y, color='#e74c3c', linewidth=2, label='ECDF (경험적 CDF)')

# KDE 기반 CDF 오버레이
cdf_kde = np.array([kde.integrate_box_1d(-np.inf, x) for x in x_grid])
axes[1].plot(x_grid, cdf_kde, color='#3498db', linewidth=2, linestyle='--', label='KDE 기반 CDF')

# P(age <= 30) 표시
cdf_at_30 = kde.integrate_box_1d(-np.inf, 30)
axes[1].axhline(y=cdf_at_30, color='gray', linestyle=':', alpha=0.7)
axes[1].axvline(x=30, color='gray', linestyle=':', alpha=0.7)
axes[1].plot(30, cdf_at_30, 'o', color='green', markersize=10, zorder=10)
axes[1].annotate(f'F(30) = {cdf_at_30:.4f}\nP(age≤30) = {cdf_at_30*100:.1f}%',
                xy=(30, cdf_at_30), xytext=(45, cdf_at_30 - 0.15),
                fontsize=12, fontweight='bold', color='green',
                arrowprops=dict(arrowstyle='->', color='green', lw=2))

axes[1].set_xlabel('나이', fontsize=13)
axes[1].set_ylabel('F(x) = P(X ≤ x)', fontsize=13)
axes[1].set_title('CDF (누적분포함수)', fontsize=16, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# CDF를 활용한 다양한 확률 계산
print("=" * 60)
print("CDF를 활용한 확률 계산")
print("=" * 60)

# P(age <= 30)
cdf_30 = kde.integrate_box_1d(-np.inf, 30)
emp_30 = (age <= 30).mean()
print(f"\n1. P(age ≤ 30) = F(30)")
print(f"   KDE 추정: {cdf_30:.4f} ({cdf_30*100:.1f}%)")
print(f"   실제 데이터: {emp_30:.4f} ({emp_30*100:.1f}%)")

# P(age > 50)
cdf_50 = kde.integrate_box_1d(-np.inf, 50)
p_gt_50 = 1 - cdf_50
emp_gt_50 = (age > 50).mean()
print(f"\n2. P(age > 50) = 1 - F(50)")
print(f"   KDE 추정: {p_gt_50:.4f} ({p_gt_50*100:.1f}%)")
print(f"   실제 데이터: {emp_gt_50:.4f} ({emp_gt_50*100:.1f}%)")

# P(20 <= age <= 40)
cdf_20 = kde.integrate_box_1d(-np.inf, 20)
cdf_40 = kde.integrate_box_1d(-np.inf, 40)
p_20_40 = cdf_40 - cdf_20
emp_20_40 = ((age >= 20) & (age <= 40)).mean()
print(f"\n3. P(20 ≤ age ≤ 40) = F(40) - F(20)")
print(f"   F(40) = {cdf_40:.4f}")
print(f"   F(20) = {cdf_20:.4f}")
print(f"   P(20 ≤ age ≤ 40) = {cdf_40:.4f} - {cdf_20:.4f} = {p_20_40:.4f} ({p_20_40*100:.1f}%)")
print(f"   실제 데이터: {emp_20_40:.4f} ({emp_20_40*100:.1f}%)")

# P(age <= 5) — 영유아 비율
cdf_5 = kde.integrate_box_1d(-np.inf, 5)
emp_5 = (age <= 5).mean()
print(f"\n4. P(age ≤ 5) = F(5) — 영유아 비율")
print(f"   KDE 추정: {cdf_5:.4f} ({cdf_5*100:.1f}%)")
print(f"   실제 데이터: {emp_5:.4f} ({emp_5*100:.1f}%)")

### 해석

- **P(age <= 30)**: 약 55%의 승객이 30세 이하였다. 젊은 승객이 과반수를 차지한다.
- **P(20 <= age <= 40)**: 20~40세 구간이 전체의 약 58%를 차지하는 핵심 연령대이다.
- **P(age > 50)**: 50세 초과 승객은 약 12%로 소수이다.
- KDE 추정값과 실제 데이터 비율이 매우 유사하여, KDE가 좋은 추정치임을 확인할 수 있다.

---

## 6단계: KDE 대역폭(Bandwidth) 영향 분석

KDE에서 **대역폭(bandwidth, $h$)**은 추정의 부드러움을 결정하는 핵심 파라미터이다.

- **좁은 대역폭 (작은 $h$)**: 데이터의 세부 구조를 포착하지만, 잡음에 민감 → **과적합(overfitting)** 위험
- **넓은 대역폭 (큰 $h$)**: 부드러운 곡선이지만, 중요한 구조를 놓칠 수 있음 → **과소적합(underfitting)** 위험
- **적정 대역폭**: 데이터의 진짜 패턴은 포착하되, 잡음은 무시하는 균형점

In [ ]:
# Bandwidth 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bandwidths = [0.5, None, 5.0]  # None = 자동 (Scott's rule)
bw_labels = ['좁은 BW (h=0.5)\n과적합 위험', '자동 BW (Scott\'s rule)\n적정 균형', '넓은 BW (h=5.0)\n과소적합 위험']
bw_colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, bw, label, color in zip(axes, bandwidths, bw_labels, bw_colors):
    # 히스토그램 (배경)
    sns.histplot(age, bins=30, stat='density', color='lightgray',
                 edgecolor='white', alpha=0.5, ax=ax)
    
    # KDE
    if bw is None:
        kde_temp = gaussian_kde(age)
    else:
        kde_temp = gaussian_kde(age, bw_method=bw / age.std())
    
    pdf_temp = kde_temp(x_grid)
    ax.plot(x_grid, pdf_temp, color=color, linewidth=2.5)
    ax.fill_between(x_grid, pdf_temp, alpha=0.15, color=color)
    
    ax.set_xlabel('나이', fontsize=12)
    ax.set_ylabel('밀도', fontsize=12)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 0.06)

fig.suptitle('KDE 대역폭(Bandwidth)에 따른 밀도 추정 변화\n'
             '너무 좁으면 잡음에 민감, 너무 넓으면 구조가 묻힘',
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

In [ ]:
# 모든 bandwidth를 한 그래프에 겹쳐 비교
fig, ax = plt.subplots(figsize=(12, 6))

# 히스토그램 배경
sns.histplot(age, bins=30, stat='density', color='lightgray',
             edgecolor='white', alpha=0.4, ax=ax, label='히스토그램')

bw_test = [0.5, 1.0, 2.0, 5.0]
colors_bw = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']

for bw, color in zip(bw_test, colors_bw):
    kde_temp = gaussian_kde(age, bw_method=bw / age.std())
    pdf_temp = kde_temp(x_grid)
    ax.plot(x_grid, pdf_temp, color=color, linewidth=2.5, label=f'BW = {bw}')

# 자동 bandwidth도 추가
kde_auto = gaussian_kde(age)
pdf_auto = kde_auto(x_grid)
ax.plot(x_grid, pdf_auto, color='black', linewidth=3, linestyle='--', label=f'자동 (Scott)')

ax.set_xlabel('나이', fontsize=14)
ax.set_ylabel('밀도', fontsize=14)
ax.set_title('대역폭(Bandwidth) 비교: 좁은 BW → 울퉁불퉁, 넓은 BW → 과도하게 부드러움',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 0.055)

plt.tight_layout()
plt.show()

### 해석

- **BW=0.5 (빨강)**: 데이터의 모든 요동을 따라가려 하여 울퉁불퉁하다. 개별 데이터 포인트의 잡음이 반영됨.
- **BW=5.0 (파랑)**: 너무 부드러워서 분포의 실제 구조(예: 영유아 봉우리, 주요 봉우리의 위치)가 사라진다.
- **자동 BW (Scott's rule, 검정 점선)**: 편향과 분산의 최적 균형을 찾아주는 이론적으로 검증된 방법이다.
- 실무에서는 보통 자동 방법을 사용하되, 도메인 지식을 활용하여 조정할 수 있다.

### Bias-Variance Tradeoff

| 대역폭 | 편향(Bias) | 분산(Variance) | 결과 |
|--------|-----------|---------------|------|
| 좁음 ($h \to 0$) | 낮음 | 높음 | 과적합 |
| 넓음 ($h \to \infty$) | 높음 | 낮음 | 과소적합 |
| 적정 | 균형 | 균형 | 최적 추정 |

---

## 종합 정리

### PDF와 CDF — 연속 확률변수 버전

$$\text{PDF} \xrightarrow{\text{적분}} \text{CDF}: \quad F(x) = \int_{-\infty}^{x} f(t) \, dt$$

$$\text{CDF} \xrightarrow{\text{미분}} \text{PDF}: \quad f(x) = \frac{d}{dx} F(x)$$

| 항목 | PDF | CDF |
|------|-----|-----|
| **표기** | $f(x)$ | $F(x) = P(X \leq x)$ |
| **y축 의미** | 밀도 (확률이 아님!) | 누적 확률 (0~1) |
| **확률 계산** | 면적(적분) | 함수값 읽기 |
| **그래프** | 연속 곡선 | S자형 단조증가 곡선 |
| **핵심 성질** | $\int f(x)dx = 1$ | $F(-\infty)=0, F(+\infty)=1$ |

### 핵심 개념 요약

1. **연속 확률변수**에서 특정 값의 확률은 항상 0이다: $P(X = x) = 0$. 구간 확률만 의미가 있다.
2. **PDF의 y축은 밀도**이다. 밀도 $\times$ 구간폭 = 확률. PDF 값이 1을 초과할 수 있다.
3. **히스토그램**은 bins 선택에 민감하지만, **KDE**는 대역폭으로 부드러움을 조절하여 더 안정적이다.
4. **CDF**는 PDF를 적분한 것이며, $F(b) - F(a) = P(a \leq X \leq b)$로 구간 확률을 바로 구할 수 있다.
5. **대역폭(bandwidth)**은 KDE의 핵심 파라미터이며, 편향-분산 트레이드오프의 좋은 예이다.